# Attention U-Net + HRNet-W32 — Prostate Tumor Segmentation (Multimodal T2W + ADC + HBV)

Builds on `unet_hrnet_tumor_seg_REVISED.ipynb` (loss fix, augmentation, discriminative LR, GroupNorm, patient-level Dice) and adds the biparametric MRI inputs:

- **ADC** and **HBV** (high b-value DWI) are now loaded from the same patient folder as T2W and stacked as extra channels — this matches PI-CAI's own official baseline protocol, which trains on all three sequences together rather than T2W alone.
- ADC/HBV are **resampled onto the T2W grid** before stacking, since they're commonly acquired at different resolution/FOV.
- Each modality gets its own normalization: T2W keeps the existing per-slice z-score; ADC is clipped to a physiological range and scaled (its intensities are literal diffusion coefficients, not arbitrary units, so z-scoring per slice would distort the signal you actually want); HBV is percentile-clipped then z-scored (its scale varies more across scanners/vendors).
- The encoder now takes **3 input channels**. Since HRNet-W32's pretrained stem already expects 3 channels (it was trained on RGB), we no longer need to average/collapse the pretrained weights down to 1 channel like the single-modality version did — the pretrained stem transfers across as-is, which is a strictly better initialization than the single-channel average.
- **`TVERSKY_ALPHA`/`TVERSKY_BETA` reset to 0.5/0.5** (balanced) by default. Your last run at `alpha=0.3, beta=0.7` traded precision for recall and lowered Dice (0.5488 → 0.5177) — since your stated goal here is maximizing Dice specifically, the balanced setting is the safer default; the recall-biased setting is still available if false negatives matter more than Dice for your writeup.

Everywhere something changed from the single-modality notebook there's a `# CHANGED:` or `# NEW:` comment.

In [ ]:
# =============================================================================
# CELL 1 — Imports & output folder
# =============================================================================
import os, gc, json, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
try:
    from torchvision.models import resnet50, ResNet50_Weights
except Exception:
    resnet50 = None
    ResNet50_Weights = None
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix

SAVE_DIR = "Unet_Hrnet_results_multimodal"   # CHANGED: separate output folder so this run never overwrites your single-modality results
for d in [SAVE_DIR,
          os.path.join(SAVE_DIR, 'checkpoints'),
          os.path.join(SAVE_DIR, 'plots'),
          os.path.join(SAVE_DIR, 'visuals')]:
    os.makedirs(d, exist_ok=True)
print(f"Output folder ready: {SAVE_DIR}/")


In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
import torch

print("Torch Version :", torch.__version__)
print("CUDA Version  :", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("GPU Count     :", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name      :", torch.cuda.get_device_name(0))


In [ ]:
import os
import torch

# ============================================================
# DATASET PATHS
# ============================================================

MRI_ROOT = [r"D:\\BT23ECE037_Kapil_FYP27\\Pi-cai_dataset\\picai_dataset_all_folds"]
MASK_ROOT = r"D:\\BT23ECE037_Kapil_FYP27\\Pi-cai_dataset\\Mask"

MODALITY_SUFFIXES = {
    "t2w": "_t2w.mha",
    "adc": "_adc.mha",
    "hbv": "_hbv.mha",
}
MODALITIES = ["t2w", "adc", "hbv"]
IN_CHANNELS = len(MODALITIES)

# ============================================================
# PROSTATE TUMOR SEGMENTATION CLASSES
# ============================================================

CLASS_NAMES = [
    "Background",
    "Tumor"
]

NUM_CLASSES = 2
TARGET_SIZE = (320, 320)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 2
EPOCHS = 50

LR = 1e-4
ENCODER_LR_MULT = 0.1
ENCODER_FREEZE_EPOCHS = 3

K_FOLDS = 5
TEST_SPLIT = 0.20
SEED = 42

# CHANGED: class weights now actually used -- previously defined but silently ignored by the
# Tversky loss path. Now feeds the Focal term of the new "combo" loss (single-weighted, per
# the double-weighting bug already fixed earlier in this project).
CLASS_WEIGHTS = torch.tensor([0.2, 0.8], dtype=torch.float32).to(DEVICE)

# CHANGED: LOSS_TYPE default -> "combo" = Tversky (imbalance-aware FP/FN trade-off) +
# class-weighted Focal (extra push on the rare tumor class), rather than Tversky alone.
LOSS_TYPE = "combo"    # "combo" | "tversky" | "dice_focal"

# CHANGED: alpha/beta nudged from your last balanced run (0.5/0.5, Dice 0.6206, recall 0.6129
# slightly behind precision 0.6285) toward recall -- mild, not the 0.3/0.7 overcorrection that
# hurt Dice on the single-modality run.
TVERSKY_ALPHA = 0.45
TVERSKY_BETA = 0.55

# NEW: relative weight of the two terms inside the "combo" loss.
TVERSKY_WEIGHT = 0.7
FOCAL_WEIGHT = 0.3

# NEW: cosine-annealing-with-warmup schedule, replacing ReduceLROnPlateau.
WARMUP_EPOCHS = 3

SAVE_PREFIX = os.path.join(SAVE_DIR, "checkpoints", "attn_unet_hrnetw32_prostate_tumor_mm")

print(
    f"Device: {DEVICE} | "
    f"Target: {TARGET_SIZE} | "
    f"Model: HRNet-W32 | "
    f"Modalities: {MODALITIES} (in_channels={IN_CHANNELS}) | "
    f"Classes: {NUM_CLASSES} | "
    f"Loss: {LOSS_TYPE} (tversky alpha={TVERSKY_ALPHA}, beta={TVERSKY_BETA}, "
    f"tversky_w={TVERSKY_WEIGHT}, focal_w={FOCAL_WEIGHT}, class_weights={CLASS_WEIGHTS.tolist()}) | "
    f"LR schedule: cosine warmup ({WARMUP_EPOCHS} warmup epochs) | "
    f"Task: Prostate Tumor Segmentation"
)


## Case scanning — CHANGED

Now requires ADC and HBV files to exist alongside T2W (same folder, same `study_id` prefix) before a case is kept. Cases missing a modality are reported and dropped, rather than silently failing later inside the Dataset. If your files use different suffixes (e.g. `_hbv_calc.mha`, `_ADC.mha`), edit `MODALITY_SUFFIXES` above and re-run this cell.

In [ ]:
patient_cases = []
skipped_missing_modality = []

for fold_path in MRI_ROOT:
    for patient_id in os.listdir(fold_path):
        patient_dir = os.path.join(fold_path, patient_id)
        if not os.path.isdir(patient_dir):
            continue

        t2w_files = [f for f in os.listdir(patient_dir) if f.endswith(MODALITY_SUFFIXES["t2w"])]
        if len(t2w_files) == 0:
            continue

        t2w_file = t2w_files[0]
        study_id = t2w_file.replace(MODALITY_SUFFIXES["t2w"], "")
        mask_path = os.path.join(MASK_ROOT, study_id + ".nii.gz")
        if not os.path.exists(mask_path):
            continue

        # NEW: build a path per modality and require all of them to exist
        modality_paths = {}
        missing = []
        for mod in MODALITIES:
            p = os.path.join(patient_dir, study_id + MODALITY_SUFFIXES[mod])
            if os.path.exists(p):
                modality_paths[mod] = p
            else:
                missing.append(mod)

        if missing:
            skipped_missing_modality.append((study_id, missing))
            continue

        patient_cases.append((patient_id, study_id, modality_paths, mask_path))   # CHANGED: img_path -> modality_paths dict

print(f"Patient cases found (all {len(MODALITIES)} modalities present): {len(patient_cases)}")
print(f"Patient cases skipped (missing a modality): {len(skipped_missing_modality)}")
if skipped_missing_modality:
    print("First few skipped, with what was missing:")
    for sid, miss in skipped_missing_modality[:10]:
        print(f"  {sid}: missing {miss}")

random.seed(SEED)
random.shuffle(patient_cases)

n_test = int(len(patient_cases) * TEST_SPLIT)
test_cases = patient_cases[:n_test]
trainval_cases = patient_cases[n_test:]

print(f"Total={len(patient_cases)}")
print(f"Train+Val={len(trainval_cases)}")
print(f"Test={len(test_cases)}")


In [ ]:
import nibabel as nib
import numpy as np

positive_cases = []
negative_cases = []

for patient_id, study_id, modality_paths, mask_path in patient_cases:
    mask = nib.load(mask_path).get_fdata()
    if np.any(mask > 0):
        positive_cases.append((patient_id, study_id, modality_paths, mask_path))
    else:
        negative_cases.append((patient_id, study_id, modality_paths, mask_path))

print("=" * 60)
print("DATASET CHECK")
print("=" * 60)
print("Total cases       :", len(patient_cases))
print("Positive cases    :", len(positive_cases))
print("Blank-mask cases  :", len(negative_cases))


## Dataset — CHANGED: loads and stacks T2W + ADC + HBV

Key points:
- ADC and HBV are **resampled onto the T2W grid** (`sitk.Resample`, T2W as the reference image) before extracting the slice — this handles the case where they were acquired at different resolution/FOV, which is common for DWI sequences.
- Per-modality normalization, not one-size-fits-all z-scoring:
  - **T2W** — per-slice z-score (unchanged from before).
  - **ADC** — clipped to `[0, 3000]` (×10⁻⁶ mm²/s, covers the physiological range for prostate tissue) then scaled to `[0, 1]`. ADC intensities are literal diffusion coefficients — z-scoring per slice would erase the fixed clinical meaning of "low ADC ≈ suspicious."
  - **HBV** — clipped to the 1st–99th percentile of the volume, then z-scored (its intensity scale varies more across scanners/vendors than ADC does).
- A **per-volume cache** avoids re-reading and re-resampling the same patient's 3 volumes for every slice — meaningfully faster than reloading from disk per `__getitem__` call given resampling is not free.

In [ ]:
# =============================================================================
# PI-CAI Multimodal Dataset (T2W + ADC + HBV)
# =============================================================================

import os
import random
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from collections import OrderedDict

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset
from tqdm import tqdm

VOLUME_CACHE_SIZE = 6   # max number of full T2W+ADC+HBV volumes kept in RAM at once, per dataset instance.
                        # Now cheap to keep since each cached volume is at target_size, not native resolution.


# CHANGED: normalization now operates on the WHOLE volume in one vectorized call, not per-slice
# in a Python loop -- that loop was what caused the MemoryError (it built 27 separate float64
# arrays before concatenating them back together). Also explicitly cast to float32 throughout,
# since np.percentile/np.clip silently upcast to float64 otherwise.

def normalize_t2w_vol(vol):
    vol = vol.astype(np.float32)
    return ((vol - vol.mean()) / (vol.std() + 1e-8)).astype(np.float32)


def normalize_adc_vol(vol, clip_min=0.0, clip_max=3000.0):
    v = np.clip(vol.astype(np.float32), clip_min, clip_max)
    return ((v - clip_min) / (clip_max - clip_min + 1e-8)).astype(np.float32)


def normalize_hbv_vol(vol):
    vol = vol.astype(np.float32)
    lo, hi = np.percentile(vol, [1, 99]).astype(np.float32)
    v = np.clip(vol, lo, hi)
    return ((v - v.mean()) / (v.std() + 1e-8)).astype(np.float32)


NORMALIZERS_VOL = {"t2w": normalize_t2w_vol, "adc": normalize_adc_vol, "hbv": normalize_hbv_vol}


class PICAITumorDataset(Dataset):

    def __init__(self, patient_cases, target_size=(320, 320), augment=False,
                 modalities=None, cache_volumes=True, max_cache_size=VOLUME_CACHE_SIZE):
        self.target_size = target_size
        self.augment = augment
        self.modalities = modalities or MODALITIES
        self.cache_volumes = cache_volumes
        self.max_cache_size = max_cache_size
        self._volume_cache = OrderedDict()
        self.slice_index = []
        self.patient_cases = patient_cases

        positive_slices = []
        negative_slices = []

        print(f"Scanning {len(patient_cases)} volumes...")

        for case in tqdm(patient_cases, desc="Building Slice Index"):
            patient_id, study_id, modality_paths, mask_path = case
            try:
                mask = nib.load(mask_path).get_fdata()
                mask = np.transpose(mask, (1, 0, 2))
                mask = (mask > 0).astype(np.uint8)
                n_slices = mask.shape[-1]
                case_info = (patient_id, study_id, modality_paths, mask_path)

                for s in range(n_slices):
                    if np.any(mask[:, :, s] > 0):
                        positive_slices.append((case_info, s))
                    else:
                        negative_slices.append((case_info, s))

                del mask
            except Exception as e:
                print(f"Skipping {study_id}: {e}")

        random.shuffle(negative_slices)
        negative_slices = negative_slices[: len(positive_slices) // 2]

        self.slice_index = positive_slices + negative_slices
        random.shuffle(self.slice_index)

        print(f"Positive slices: {len(positive_slices)}")
        print(f"Negative slices: {len(negative_slices)}")
        print(f"Total slices: {len(self.slice_index)}")
        print(f"Modalities: {self.modalities} | Augmentation: {'ON' if self.augment else 'OFF'} | "
              f"Volume cache: {'ON (max ' + str(self.max_cache_size) + ' volumes)' if self.cache_volumes else 'OFF'}")

    def __len__(self):
        return len(self.slice_index)

    def _augment(self, img, msk):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[-1])
            msk = torch.flip(msk, dims=[-1])
        if random.random() < 0.3:
            k = random.choice([1, 2, 3])
            img = torch.rot90(img, k, dims=[-2, -1])
            msk = torch.rot90(msk, k, dims=[-2, -1])
        if random.random() < 0.3:
            img = img + torch.randn_like(img) * 0.05
        return img, msk

    def _cache_get(self, study_id):
        if study_id not in self._volume_cache:
            return None
        self._volume_cache.move_to_end(study_id)
        return self._volume_cache[study_id]

    def _cache_put(self, study_id, value):
        self._volume_cache[study_id] = value
        self._volume_cache.move_to_end(study_id)
        while len(self._volume_cache) > self.max_cache_size:
            self._volume_cache.popitem(last=False)

    def _downsample_volume(self, vol_hwd, mode):
        # CHANGED: shrink a [H,W,D] volume to target_size in ONE interpolate call, at native
        # dtype float32, BEFORE any normalization touches it. This is the key memory fix --
        # everything downstream (normalization, stacking, caching) now operates on small
        # target_size arrays instead of 1024x1024-scale ones.
        t = torch.from_numpy(vol_hwd.astype(np.float32)).permute(2, 0, 1).unsqueeze(1)  # [D,1,H,W]
        t = F.interpolate(t, size=self.target_size, mode=mode,
                           align_corners=False if mode == 'bilinear' else None)
        return t.squeeze(1).permute(1, 2, 0).numpy()  # [h,w,D]

    def _load_and_stack_volume(self, study_id, modality_paths, mask_path):
        if self.cache_volumes:
            cached = self._cache_get(study_id)
            if cached is not None:
                return cached

        t2w_itk = sitk.ReadImage(modality_paths["t2w"])
        t2w_arr = sitk.GetArrayFromImage(t2w_itk).astype(np.float32)
        t2w_arr = np.transpose(t2w_arr, (1, 2, 0))  # [H,W,D], still native resolution here

        stacked = []
        for mod in self.modalities:
            if mod == "t2w":
                raw_vol = t2w_arr
            else:
                mod_itk = sitk.ReadImage(modality_paths[mod])
                try:
                    resampled = sitk.Resample(
                        mod_itk, t2w_itk, sitk.Transform(), sitk.sitkLinear, 0.0, mod_itk.GetPixelID()
                    )
                    raw_vol = sitk.GetArrayFromImage(resampled).astype(np.float32)
                    raw_vol = np.transpose(raw_vol, (1, 2, 0))
                except Exception as e:
                    print(f"  Resample failed for {study_id} [{mod}]: {e} -> using zeros")
                    raw_vol = np.zeros_like(t2w_arr)

            # CHANGED: downsample THIS modality immediately, before normalization, so we never
            # hold a native-resolution multi-slice float array for long.
            small_vol = self._downsample_volume(raw_vol, mode='bilinear')
            del raw_vol
            normed = NORMALIZERS_VOL[mod](small_vol)
            stacked.append(normed)

        stacked = np.stack(stacked, axis=0).astype(np.float32)  # [C,h,w,D], already small

        mask = nib.load(mask_path).get_fdata()
        mask = np.transpose(mask, (1, 0, 2))
        mask = (mask > 0).astype(np.float32)
        mask = self._downsample_volume(mask, mode='nearest').astype(np.int64)  # [h,w,D]

        result = (stacked, mask)
        if self.cache_volumes:
            self._cache_put(study_id, result)
        return result

    def __getitem__(self, idx):
        case_info, s = self.slice_index[idx]
        patient_id, study_id, modality_paths, mask_path = case_info

        stacked, mask = self._load_and_stack_volume(study_id, modality_paths, mask_path)

        min_z = min(stacked.shape[-1], mask.shape[-1])
        s = min(s, min_z - 1)

        img = torch.tensor(stacked[:, :, :, s], dtype=torch.float32)   # [C,h,w] -- already target_size
        msk = torch.tensor(mask[:, :, s], dtype=torch.long)             # [h,w]

        if self.augment:
            img, msk = self._augment(img, msk)

        return (img, msk, study_id, s)


In [ ]:
print("\nBuilding datasets...")

trainval_dataset = PICAITumorDataset(trainval_cases, TARGET_SIZE, augment=True)
test_dataset = PICAITumorDataset(test_cases, TARGET_SIZE, augment=False)


In [ ]:
# =============================================================================
# CELL 4 — Quick Dataset Sanity Check (Tumor Segmentation, multimodal)
# =============================================================================

COLORS = np.array([
    [0,   0,   0],      # Background (Black)
    [255, 0,   0]       # Tumor (Red)
], dtype=np.uint8)

idx = min(2711, len(trainval_dataset) - 1)   # CHANGED: guard against dataset size shrinking after modality filtering
img, mask, study_id, s = trainval_dataset[idx]

print(f"Study ID : {study_id}")
print(f"Slice    : {s}")
print(f"Image : shape={tuple(img.shape)} (channels={MODALITIES})")
print(f"Mask  : shape={tuple(mask.shape)} unique={mask.unique().tolist()}")
print(f"Tumor Pixels = {(mask == 1).sum().item()}")

n_mod = img.shape[0]
fig, axes = plt.subplots(1, n_mod + 1, figsize=(5 * (n_mod + 1), 5))

for i, mod in enumerate(MODALITIES):
    axes[i].imshow(img[i].numpy(), cmap="gray")
    axes[i].set_title(mod.upper())
    axes[i].axis("off")

axes[-1].imshow(img[0].numpy(), cmap="gray")
axes[-1].imshow(mask.numpy(), alpha=0.5, cmap="Reds")
axes[-1].set_title("T2W + Tumor Overlay")
axes[-1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "plots", "tumor_sample_multimodal.png"), dpi=150, bbox_inches="tight")
plt.show()


## Model — CHANGED

- `AttentionUNetHRNetW32` now takes `in_channels` (default 3). When `in_channels == 3` and `pretrained=True`, the pretrained stem is used **as-is** — HRNet's pretrained `conv1` already expects 3 channels, so nothing needs to be averaged or discarded this time, unlike the single-modality (`in_channels=1`) version.
- Decoder still uses `GroupNorm` throughout (unchanged from the previous revision).

In [ ]:
import torch
import timm
import torchvision
import torch.nn as nn
import torch.nn.functional as F


def make_gn(channels, max_groups=8):
    g = min(max_groups, channels)
    while g > 1 and channels % g != 0:
        g -= 1
    return nn.GroupNorm(num_groups=max(1, g), num_channels=channels)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True), make_gn(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True), make_gn(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True), make_gn(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(self.W_g(g) + self.W_x(x)))


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.attention = AttentionGate(F_g=out_ch, F_l=skip_ch, F_int=max(1, out_ch // 2))
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            make_gn(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            make_gn(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        skip_att = self.attention(x, skip)
        return self.conv(torch.cat([skip_att, x], dim=1))


def build_stem_conv1(pretrained_conv1, in_channels, pretrained):
    # NEW: generalized stem builder. If in_channels==3 and pretrained, reuse the pretrained
    # RGB weights directly (best case: no information thrown away, unlike averaging to 1 channel).
    # For any other channel count, fall back to averaging + tiling as an approximation.
    new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False)
    if pretrained and pretrained_conv1 is not None:
        with torch.no_grad():
            w = pretrained_conv1.weight  # [64, 3, 3, 3]
            if in_channels == 3:
                new_conv1.weight.copy_(w)
            elif in_channels == 1:
                new_conv1.weight.copy_(w.mean(dim=1, keepdim=True))
            else:
                avg = w.mean(dim=1, keepdim=True)
                new_conv1.weight.copy_(avg.repeat(1, in_channels, 1, 1))
    return new_conv1


class AttentionUNetHRNetW32(nn.Module):
    def __init__(self, num_classes=2, pretrained=False, in_channels=3):   # CHANGED: in_channels param
        super().__init__()

        self.encoder = timm.create_model('hrnet_w32', pretrained=pretrained, features_only=True)

        pretrained_conv1 = self.encoder.conv1 if pretrained else None
        self.encoder.conv1 = build_stem_conv1(pretrained_conv1, in_channels, pretrained)

        with torch.no_grad():
            dummy = torch.randn(1, in_channels, 320, 320)
            feats = self.encoder(dummy)
            channels = [f.shape[1] for f in feats]

        ch1, ch2, ch3, ch4, ch5 = channels

        self.dec4 = DecoderBlock(in_ch=ch5, skip_ch=ch4, out_ch=ch4)
        self.dec3 = DecoderBlock(in_ch=ch4, skip_ch=ch3, out_ch=ch3)
        self.dec2 = DecoderBlock(in_ch=ch3, skip_ch=ch2, out_ch=ch2)
        self.dec1 = DecoderBlock(in_ch=ch2, skip_ch=ch1, out_ch=ch1)

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(ch1, 32, kernel_size=2, stride=2),
            make_gn(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            make_gn(32), nn.ReLU(inplace=True)
        )
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        input_shape = x.shape[-2:]
        e1, e2, e3, e4, e5 = self.encoder(x)

        d4 = self.dec4(e5, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)

        out = self.final_up(d1)
        if out.shape[-2:] != input_shape:
            out = F.interpolate(out, size=input_shape, mode='bilinear', align_corners=False)

        return self.head(out)


# Model initialization sanity check
model = AttentionUNetHRNetW32(num_classes=NUM_CLASSES, pretrained=False, in_channels=IN_CHANNELS).to(DEVICE)
print(f"HRNet-W32 Encoder Model Loaded ({IN_CHANNELS}-channel input). Total params: {sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    dummy_out = model(torch.randn(1, IN_CHANNELS, 320, 320).to(DEVICE))
    print(f"Verified Output Shape: {tuple(dummy_out.shape)}")


In [ ]:
# =============================================================================
# CELL 6 — Loss (fixed weighting + Focal Tversky + Combo option) & batch Dice
# =============================================================================

class DiceLoss(nn.Module):
    def __init__(self, n_classes, weights=None, smooth=1e-6):
        super().__init__()
        self.n_classes = n_classes; self.weights = weights; self.smooth = smooth

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1)
        t_oh = F.one_hot(targets, self.n_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        inter = torch.sum(probs * t_oh, dims)
        card = torch.sum(probs + t_oh, dims)
        dc = 1.0 - (2.0 * inter + self.smooth) / (card + self.smooth)
        return (dc * self.weights).mean() if self.weights is not None else dc.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, class_weights=None):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.class_weights = class_weights

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.class_weights, reduction='none')
        focal = self.alpha * (1 - torch.exp(-ce)) ** self.gamma * ce
        return focal.mean()


class DiceFocalLossFixed(nn.Module):
    def __init__(self, n_classes, class_weights=None):
        super().__init__()
        self.dice = DiceLoss(n_classes, weights=class_weights)
        self.focal = FocalLoss(alpha=0.25, gamma=2.0, class_weights=None)

    def forward(self, inputs, targets):
        return self.dice(inputs, targets) + self.focal(inputs, targets)


class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1)[:, 1]
        targets = targets.float()
        TP = (probs * targets).sum()
        FP = (probs * (1 - targets)).sum()
        FN = ((1 - probs) * targets).sum()
        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return (1 - tversky) ** self.gamma


class ComboLoss(nn.Module):
    # NEW: Tversky (handles the FP/FN trade-off via alpha/beta) + class-weighted Focal
    # (gives the class-weight knob an actual effect, since Tversky alone ignores CLASS_WEIGHTS).
    # Class weights applied ONLY here, in the Focal term -- single-weighted, not double-applied.
    def __init__(self, tversky_alpha, tversky_beta, class_weights,
                 tversky_weight=0.7, focal_weight=0.3, tversky_gamma=0.75,
                 focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.tversky = FocalTverskyLoss(alpha=tversky_alpha, beta=tversky_beta, gamma=tversky_gamma)
        self.focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, class_weights=class_weights)
        self.tversky_weight = tversky_weight
        self.focal_weight = focal_weight

    def forward(self, inputs, targets):
        return (self.tversky_weight * self.tversky(inputs, targets) +
                self.focal_weight * self.focal(inputs, targets))


def build_criterion(loss_type, n_classes, class_weights):
    # CHANGED: added "combo" option, now the default (see LOSS_TYPE in the config cell).
    if loss_type == "combo":
        return ComboLoss(
            tversky_alpha=TVERSKY_ALPHA, tversky_beta=TVERSKY_BETA,
            class_weights=class_weights,
            tversky_weight=TVERSKY_WEIGHT, focal_weight=FOCAL_WEIGHT,
        )
    elif loss_type == "tversky":
        return FocalTverskyLoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA, gamma=0.75)
    elif loss_type == "dice_focal":
        return DiceFocalLossFixed(n_classes=n_classes, class_weights=class_weights)
    else:
        raise ValueError(f"Unknown LOSS_TYPE: {loss_type}")


def get_batch_dice(preds, targets, num_classes):
    pred_cls = torch.argmax(F.softmax(preds, dim=1), dim=1)
    return [
        ((2 * (pred_cls == c).float() * (targets == c).float()).sum() + 1e-6) /
        ((pred_cls == c).float().sum() + (targets == c).float().sum() + 1e-6)
        for c in range(num_classes)
    ]


In [ ]:
# =============================================================================
# CELL 7 — Single-fold training (discriminative LR + cosine warmup + encoder warm-up + early stopping)
# =============================================================================
import math

def make_cosine_warmup_lambda(total_epochs, warmup_epochs):
    # NEW: linear warmup for `warmup_epochs`, then cosine decay to 0 over the remaining epochs.
    # Returned as a multiplier on each param group's base LR (so encoder/decoder keep their ratio).
    def _lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, (total_epochs - warmup_epochs))
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return _lambda


def train_one_fold(model, train_loader, val_loader, epochs, save_prefix, fold_idx):
    model = model.to(DEVICE)
    criterion = build_criterion(LOSS_TYPE, NUM_CLASSES, CLASS_WEIGHTS)

    optimizer = optim.Adam([
        {'params': model.encoder.parameters(), 'lr': LR * ENCODER_LR_MULT},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('encoder')], 'lr': LR}
    ])

    # CHANGED: cosine annealing with linear warmup, replacing ReduceLROnPlateau.
    # Both param groups follow the same lambda, so the encoder stays at 1/10th the decoder's LR
    # throughout -- only the shape of the decay changes, not the encoder/decoder ratio.
    cosine_lambda = make_cosine_warmup_lambda(total_epochs=epochs, warmup_epochs=WARMUP_EPOCHS)
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=[cosine_lambda, cosine_lambda])

    for p in model.encoder.parameters():
        p.requires_grad_(False)
    encoder_frozen = True

    history = {
        'train_loss': [], 'val_loss': [],
        **{f'train_dice_{n.lower()}': [] for n in CLASS_NAMES},
        **{f'val_dice_{n.lower()}': [] for n in CLASS_NAMES},
    }
    best_val_loss = float('inf'); best_epoch = 0

    early_stop_patience = 10
    patience_counter = 0

    for epoch in range(epochs):
        if encoder_frozen and epoch >= ENCODER_FREEZE_EPOCHS:
            for p in model.encoder.parameters():
                p.requires_grad_(True)
            encoder_frozen = False
            print(f"    Encoder unfrozen at epoch {epoch+1}")

        model.train()
        tr_loss = 0.0; tr_dices = [0.0] * NUM_CLASSES
        loop = tqdm(train_loader, leave=False, desc=f"Fold {fold_idx+1} | Ep {epoch+1}/{epochs} TRAIN")

        for imgs, masks, pids, sls in loop:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, masks)
            loss.backward(); optimizer.step()
            bd = get_batch_dice(out.detach(), masks, NUM_CLASSES)
            tr_loss += loss.item()
            for c in range(NUM_CLASSES):
                tr_dices[c] += bd[c].item() if hasattr(bd[c], 'item') else bd[c]
            loop.set_postfix(loss=f"{loss.item():.3f}",
                              **{CLASS_NAMES[c]: f"{bd[c]:.2f}" for c in range(1, NUM_CLASSES)})

        n_tr = len(train_loader)
        avg_tr_loss = tr_loss / n_tr
        avg_tr_dices = [d / n_tr for d in tr_dices]

        model.eval()
        va_loss = 0.0; va_dices = [0.0] * NUM_CLASSES
        with torch.no_grad():
            for imgs, masks, pids, sls in tqdm(val_loader, leave=False, desc=f"Fold {fold_idx+1} | Ep {epoch+1}/{epochs} VAL"):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                out = model(imgs)
                va_loss += criterion(out, masks).item()
                bd = get_batch_dice(out, masks, NUM_CLASSES)
                for c in range(NUM_CLASSES):
                    va_dices[c] += bd[c].item() if hasattr(bd[c], 'item') else bd[c]

        n_va = len(val_loader)
        avg_va_loss = va_loss / n_va
        avg_va_dices = [d / n_va for d in va_dices]

        print(f"\n  Fold {fold_idx+1} | Epoch {epoch+1}/{epochs}")
        print(f"    TRAIN loss={avg_tr_loss:.4f}  " +
              "  ".join(f"{n}:{avg_tr_dices[i]:.3f}" for i, n in enumerate(CLASS_NAMES)))
        print(f"    VAL   loss={avg_va_loss:.4f}  " +
              "  ".join(f"{n}:{avg_va_dices[i]:.3f}" for i, n in enumerate(CLASS_NAMES)))

        # CHANGED: cosine scheduler steps once per epoch based on epoch index, not on val_loss.
        old_lr = optimizer.param_groups[-1]['lr']
        scheduler.step()
        new_lr = optimizer.param_groups[-1]['lr']
        print(f"    Epoch {epoch+1}: LR (decoder group) = {new_lr:.2e} (was {old_lr:.2e})")

        if avg_va_loss < best_val_loss:
            best_val_loss = avg_va_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f"{save_prefix}_fold{fold_idx+1}_best.pth")
            print(f"    Best model saved (val_loss={best_val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"    Early stopping counter: {patience_counter} out of {early_stop_patience}")
            if patience_counter >= early_stop_patience:
                print("    Early stopping triggered! Moving to next fold.")
                break

        if (epoch + 1) % 5 == 0:
            ck = f"{save_prefix}_fold{fold_idx+1}_epoch{epoch+1}.pth"
            torch.save(model.state_dict(), ck)
            print(f"    Checkpoint: {ck}")

        history['train_loss'].append(avg_tr_loss)
        history['val_loss'].append(avg_va_loss)
        for i, n in enumerate(CLASS_NAMES):
            history[f'train_dice_{n.lower()}'].append(avg_tr_dices[i])
            history[f'val_dice_{n.lower()}'].append(avg_va_dices[i])

        with open(f"{save_prefix}_fold{fold_idx+1}_metrics.json", 'w') as f:
            json.dump(history, f)

        torch.cuda.empty_cache(); gc.collect()

    return history, best_val_loss, best_epoch


In [ ]:
# =============================================================================
# CELL 8 — K-Fold Runner (PI-CAI Tumor Segmentation, multimodal)
# =============================================================================

from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import shutil
import gc
import json


def run_kfold(trainval_cases, k=K_FOLDS, epochs=EPOCHS, save_prefix=SAVE_PREFIX):
    kf = KFold(n_splits=k, shuffle=True, random_state=SEED)
    case_arr = np.array(trainval_cases, dtype=object)

    fold_histories = []
    fold_best = []

    global_best_dice = 0
    global_best_path = None

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(case_arr)):
        print("\n" + "=" * 70)
        print(f"FOLD {fold_idx+1}/{k}")
        print(f"Train Cases = {len(tr_idx)}")
        print(f"Val Cases   = {len(va_idx)}")
        print("=" * 70)

        tr_ds = PICAITumorDataset(case_arr[tr_idx].tolist(), TARGET_SIZE, augment=True)
        va_ds = PICAITumorDataset(case_arr[va_idx].tolist(), TARGET_SIZE, augment=False)

        tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

        model = AttentionUNetHRNetW32(num_classes=2, pretrained=True, in_channels=IN_CHANNELS).to(DEVICE)   # CHANGED

        history, best_loss, best_ep = train_one_fold(model, tr_loader, va_loader, epochs, save_prefix, fold_idx)

        fold_histories.append(history)
        tumor_dice = history["val_dice_tumor"][best_ep - 1]

        fold_best.append({
            "fold": fold_idx + 1,
            "best_epoch": best_ep,
            "val_loss": best_loss,
            "tumor_dice": tumor_dice
        })

        if tumor_dice > global_best_dice:
            global_best_dice = tumor_dice
            global_best_path = f"{save_prefix}_fold{fold_idx+1}_best.pth"

        del model, tr_ds, va_ds
        torch.cuda.empty_cache(); gc.collect()

    print("\n")
    print("=" * 70)
    print("K-FOLD SUMMARY")
    print("=" * 70)

    for fd in fold_best:
        print(f"Fold {fd['fold']} | Epoch {fd['best_epoch']} | Loss={fd['val_loss']:.4f} | Tumor Dice={fd['tumor_dice']:.4f}")

    mean_dice = np.mean([x["tumor_dice"] for x in fold_best])
    std_dice = np.std([x["tumor_dice"] for x in fold_best])

    print("\n")
    print(f"Mean Tumor Dice = {mean_dice:.4f} \u00b1 {std_dice:.4f}")

    summary = {
        "per_fold": fold_best,
        "mean_tumor_dice": float(mean_dice),
        "std_tumor_dice": float(std_dice),
        "loss_type": LOSS_TYPE,
        "tversky_alpha": TVERSKY_ALPHA,
        "tversky_beta": TVERSKY_BETA,
        "tversky_weight": TVERSKY_WEIGHT,   # NEW
        "focal_weight": FOCAL_WEIGHT,       # NEW
        "class_weights": CLASS_WEIGHTS.tolist(),  # NEW
        "warmup_epochs": WARMUP_EPOCHS,     # NEW
        "modalities": MODALITIES,
    }

    with open(f"{save_prefix}_kfold_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    best_dest = os.path.join(SAVE_DIR, "best_model.pth")
    shutil.copy(global_best_path, best_dest)

    print("\nBest Model:")
    print(best_dest)

    return fold_histories, summary, best_dest


In [ ]:
if __name__ == "__main__":
    fold_histories, kfold_summary, best_model_path = run_kfold(trainval_cases)
    print(f"\nBest model: {best_model_path}")


## Test evaluation

Unchanged in structure from the single-modality revision — reports pixel-level per-slice-avg Dice and the patient-level macro Dice side by side, and supports TTA + connected-component filtering. Works as-is since it only reads whatever channel count `imgs` has.

In [ ]:
import glob
from collections import defaultdict

# =============================================================================
# CELL 10 — TEST EVALUATION (PI-CAI TUMOR SEGMENTATION, multimodal)
# =============================================================================

COLORS = np.array([
    [0,   0,   0],
    [255, 0,   0]
], dtype=np.uint8)


def remove_small_components(pred_mask, min_size=15):
    labeled, n = ndimage.label(pred_mask)
    if n == 0:
        return pred_mask
    sizes = ndimage.sum(pred_mask, labeled, range(1, n + 1))
    keep = np.zeros_like(pred_mask)
    for i, size in enumerate(sizes, start=1):
        if size >= min_size:
            keep[labeled == i] = 1
    return keep


def predict_logits(model, imgs, tta=False):
    out = model(imgs)
    if tta:
        flipped = torch.flip(imgs, dims=[-1])
        out_flipped = model(flipped)
        out_flipped = torch.flip(out_flipped, dims=[-1])
        out = (out + out_flipped) / 2
    return out


def evaluate_on_test(
    model_path,
    test_dataset,
    num_vis_patients=5,
    model_name="Unet_HRNetW32_Prostate_Tumor_MM",
    use_tta=False,
    use_cc_filter=False,
    cc_min_size=15,
):
    vis_dir = os.path.join(SAVE_DIR, "visuals", model_name)
    os.makedirs(vis_dir, exist_ok=True)

    model = AttentionUNetHRNetW32(num_classes=2, pretrained=False, in_channels=IN_CHANNELS).to(DEVICE)   # CHANGED
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    print(f"\nLoaded model from:\n{model_path}")
    print(f"TTA: {use_tta} | Connected-component filter: {use_cc_filter} (min_size={cc_min_size})")

    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    all_patients = list(set(case[0] for case in test_dataset.patient_cases))
    random.seed(SEED)
    vis_patients = set(random.sample(all_patients, min(num_vis_patients, len(all_patients))))
    print(f"\nVisualizing patients:\n{vis_patients}")

    total_dice = np.zeros(NUM_CLASSES, dtype=np.float64)
    total_slices = 0
    patient_stats = defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0})

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            imgs, masks, study_ids, sls = batch
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            study_id = study_ids[0]
            sl = sls[0].item()
            patient_id = study_id.split("_")[0]

            outputs = predict_logits(model, imgs, tta=use_tta)
            preds = torch.argmax(outputs, dim=1)

            preds_np = preds[0].cpu().numpy()
            if use_cc_filter:
                preds_np = remove_small_components(preds_np, min_size=cc_min_size)
            masks_np = masks[0].cpu().numpy()

            for c in range(NUM_CLASSES):
                pred_c = (preds_np == c).astype(np.float64)
                true_c = (masks_np == c).astype(np.float64)
                dice = (2 * (pred_c * true_c).sum() + 1e-6) / (pred_c.sum() + true_c.sum() + 1e-6)
                total_dice[c] += dice
            total_slices += 1

            tp = np.logical_and(preds_np == 1, masks_np == 1).sum()
            fp = np.logical_and(preds_np == 1, masks_np == 0).sum()
            fn = np.logical_and(preds_np == 0, masks_np == 1).sum()
            patient_stats[patient_id]["TP"] += tp
            patient_stats[patient_id]["FP"] += fp
            patient_stats[patient_id]["FN"] += fn

            if patient_id in vis_patients:
                has_tumor = (masks_np.max() > 0) or (preds_np.max() > 0)
                if has_tumor:
                    img_np = imgs[0, 0].cpu().numpy()   # CHANGED: channel 0 = T2W, used as the display background
                    bg = ((img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8) * 255).astype(np.uint8)
                    bg_rgb = np.stack([bg] * 3, axis=-1)
                    alpha = 0.45
                    gt_overlay = ((1 - alpha) * bg_rgb + alpha * COLORS[masks_np]).astype(np.uint8)
                    pred_overlay = ((1 - alpha) * bg_rgb + alpha * COLORS[preds_np]).astype(np.uint8)

                    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                    axes[0].imshow(bg, cmap="gray"); axes[0].set_title(f"{study_id}\nSlice {sl}")
                    axes[1].imshow(gt_overlay); axes[1].set_title("Ground Truth")
                    axes[2].imshow(pred_overlay); axes[2].set_title("Prediction")
                    for ax in axes: ax.axis("off")
                    plt.tight_layout()
                    plt.savefig(os.path.join(vis_dir, f"{study_id}_slice_{sl:03d}.png"), dpi=150, bbox_inches="tight")
                    plt.close()

    avg_dice = total_dice / total_slices
    background_dice = float(avg_dice[0])
    pixel_tumor_dice = float(avg_dice[1])

    patient_dices = []
    for pid, s in patient_stats.items():
        TP, FP, FN = s["TP"], s["FP"], s["FN"]
        d = (2 * TP + 1e-6) / (2 * TP + FP + FN + 1e-6)
        patient_dices.append(d)
    patient_macro_dice = float(np.mean(patient_dices))
    patient_macro_std = float(np.std(patient_dices))

    print("\n")
    print("=" * 60)
    print("TEST RESULTS")
    print("=" * 60)
    print(f"Background Dice (pixel, per-slice avg) : {background_dice:.4f}")
    print(f"Tumor Dice      (pixel, per-slice avg) : {pixel_tumor_dice:.4f}")
    print(f"Tumor Dice      (patient-level macro)  : {patient_macro_dice:.4f} \u00b1 {patient_macro_std:.4f}")
    print("=" * 60)

    metrics = {
        "background_dice_pixel": background_dice,
        "tumor_dice_pixel_per_slice_avg": pixel_tumor_dice,
        "tumor_dice_patient_macro": patient_macro_dice,
        "tumor_dice_patient_macro_std": patient_macro_std,
        "use_tta": use_tta,
        "use_cc_filter": use_cc_filter,
        "modalities": MODALITIES,
    }

    with open(os.path.join(SAVE_DIR, "test_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    return avg_dice, pixel_tumor_dice, patient_macro_dice


try:
    best_model_path
except NameError:
    candidate = os.path.join(SAVE_DIR, "best_model.pth")
    if os.path.exists(candidate):
        best_model_path = candidate
    else:
        ckps = glob.glob(os.path.join(SAVE_DIR, "checkpoints", "*_fold*_best.pth"))
        if len(ckps):
            best_model_path = sorted(ckps, key=os.path.getmtime)[-1]
        else:
            raise RuntimeError("No trained model found.")

if not hasattr(test_dataset, "patient_cases"):
    test_dataset.patient_cases = patient_cases

avg_dice, pixel_tumor_dice, patient_macro_dice = evaluate_on_test(
    best_model_path, test_dataset, use_tta=False, use_cc_filter=False
)
print(f"\nBaseline — pixel Dice = {pixel_tumor_dice:.4f} | patient macro Dice = {patient_macro_dice:.4f}")


In [ ]:
avg_dice_tta, pixel_tumor_dice_tta, patient_macro_dice_tta = evaluate_on_test(
    best_model_path, test_dataset,
    model_name="Unet_HRNetW32_Prostate_Tumor_MM_TTA",
    use_tta=True, use_cc_filter=True, cc_min_size=15
)
print(f"\nTTA + CC-filter — pixel Dice = {pixel_tumor_dice_tta:.4f} | patient macro Dice = {patient_macro_dice_tta:.4f}")


In [ ]:
def ensemble_predict(models, imgs, tta=False):
    probs_sum = None
    for m in models:
        logits = predict_logits(m, imgs, tta=tta)
        probs = F.softmax(logits, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(models)


def evaluate_ensemble(fold_ckpt_paths, test_dataset, use_tta=True, use_cc_filter=True, cc_min_size=15):
    models = []
    for p in fold_ckpt_paths:
        m = AttentionUNetHRNetW32(num_classes=2, pretrained=False, in_channels=IN_CHANNELS).to(DEVICE)   # CHANGED
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append(m)
    print(f"Loaded {len(models)} fold models for ensembling.")

    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    patient_stats = defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0})
    total_dice = 0.0; total_slices = 0

    with torch.no_grad():
        for imgs, masks, study_ids, sls in tqdm(test_loader, desc="Ensemble testing"):
            imgs = imgs.to(DEVICE); masks_np = masks[0].numpy()
            patient_id = study_ids[0].split("_")[0]

            probs = ensemble_predict(models, imgs, tta=use_tta)
            preds_np = torch.argmax(probs, dim=1)[0].cpu().numpy()
            if use_cc_filter:
                preds_np = remove_small_components(preds_np, min_size=cc_min_size)

            pred_c = (preds_np == 1).astype(np.float64); true_c = (masks_np == 1).astype(np.float64)
            total_dice += (2 * (pred_c * true_c).sum() + 1e-6) / (pred_c.sum() + true_c.sum() + 1e-6)
            total_slices += 1

            tp = np.logical_and(preds_np == 1, masks_np == 1).sum()
            fp = np.logical_and(preds_np == 1, masks_np == 0).sum()
            fn = np.logical_and(preds_np == 0, masks_np == 1).sum()
            patient_stats[patient_id]["TP"] += tp
            patient_stats[patient_id]["FP"] += fp
            patient_stats[patient_id]["FN"] += fn

    pixel_dice = total_dice / total_slices
    patient_dices = [
        (2 * s["TP"] + 1e-6) / (2 * s["TP"] + s["FP"] + s["FN"] + 1e-6)
        for s in patient_stats.values()
    ]
    patient_macro = float(np.mean(patient_dices))

    print(f"\nEnsemble ({len(models)} folds) — pixel Dice = {pixel_dice:.4f} | patient macro Dice = {patient_macro:.4f}")
    return pixel_dice, patient_macro


fold_ckpts = sorted(glob.glob(f"{SAVE_PREFIX}_fold*_best.pth"))
print(f"Found {len(fold_ckpts)} fold checkpoints:", fold_ckpts)

if len(fold_ckpts) > 1:
    evaluate_ensemble(fold_ckpts, test_dataset, use_tta=True, use_cc_filter=True, cc_min_size=15)
else:
    print("Need more than one fold checkpoint to ensemble — run/complete the k-fold training above first.")


In [ ]:
print(best_model_path)


In [ ]:
def plot_confusion_matrix(model_path, test_dataset, model_name='AttentionUNet_HRNetW32_MM'):
    model = AttentionUNetHRNetW32(num_classes=NUM_CLASSES, pretrained=False, in_channels=IN_CHANNELS).to(DEVICE)   # CHANGED
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for imgs, masks, _, _ in tqdm(test_loader, desc='Confusion matrix'):
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            if masks.ndim == 4:
                masks = masks.squeeze(1)
            logits = model(imgs)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.reshape(-1).cpu().numpy())
            all_targets.extend(masks.reshape(-1).cpu().numpy())

    cm = confusion_matrix(all_targets, all_preds, labels=range(NUM_CLASSES))
    cm_n = cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    print("\nConfusion Matrix:")
    print(cm)

    if NUM_CLASSES == 2:
        TN = cm[0, 0]; FP = cm[0, 1]; FN = cm[1, 0]; TP = cm[1, 1]
        accuracy = (TP + TN) / (TP + TN + FP + FN + 1e-8)
        precision = TP / (TP + FP + 1e-8)
        recall = TP / (TP + FN + 1e-8)
        dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
        iou = TP / (TP + FP + FN + 1e-8)

        print("\n========== Segmentation Metrics ==========")
        print(f"Accuracy  : {accuracy:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"Dice      : {dice:.4f}   (pixel-level micro-average, same definition as before)")
        print(f"IoU       : {iou:.4f}")
        print("==========================================")

    print("\nPer-class Recall:")
    for i, class_name in enumerate(CLASS_NAMES):
        print(f"{class_name}: {cm_n[i, i]:.4f}")

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_n, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={'label': 'Recall (row-normalised)'})
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.title(f'Prostate Tumor Confusion Matrix\n{model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()

    save_path = os.path.join(SAVE_DIR, 'plots', 'confusion_matrix.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\nSaved: {save_path}")
    plt.show()

    return cm, cm_n


In [ ]:
cm, cm_n = plot_confusion_matrix(
    model_path=best_model_path,
    test_dataset=test_dataset,
    model_name="AttentionUNet_HRNetW32_MM"
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F


def visualize_prediction(model, dataset, sample_idx, device=DEVICE):
    model.eval()
    img, mask, study_id, slice_no = dataset[sample_idx]
    x = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(x)
        pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    t2w_img = img[0].cpu().numpy()   # CHANGED: channel 0 = T2W, used for display
    mask = mask.cpu().numpy()

    img_disp = (t2w_img - t2w_img.min()) / (t2w_img.max() - t2w_img.min() + 1e-8)
    rgb = np.stack([img_disp] * 3, axis=-1)

    gt_overlay = rgb.copy(); gt_overlay[mask == 1] = [0, 1, 0]
    pred_overlay = rgb.copy(); pred_overlay[pred == 1] = [1, 0, 0]

    diff = np.zeros((*mask.shape, 3))
    TP = (pred == 1) & (mask == 1)
    FP = (pred == 1) & (mask == 0)
    FN = (pred == 0) & (mask == 1)
    diff[TP] = [0, 1, 0]; diff[FP] = [1, 0, 0]; diff[FN] = [0, 0, 1]

    inter = np.logical_and(pred, mask).sum()
    dice = (2 * inter + 1e-6) / (pred.sum() + mask.sum() + 1e-6)

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    axs[0, 0].imshow(img_disp, cmap='gray'); axs[0, 0].set_title("T2W")
    axs[0, 1].imshow(mask, cmap='gray'); axs[0, 1].set_title("Ground Truth")
    axs[0, 2].imshow(pred, cmap='gray'); axs[0, 2].set_title("Prediction")
    axs[1, 0].imshow(gt_overlay); axs[1, 0].set_title("T2W + GT")
    axs[1, 1].imshow(pred_overlay); axs[1, 1].set_title("T2W + Prediction")
    axs[1, 2].imshow(diff); axs[1, 2].set_title("Difference\nGreen=TP  Red=FP  Blue=FN")
    for ax in axs.ravel(): ax.axis("off")

    plt.suptitle(f"Study ID : {study_id}\nSlice : {slice_no}\nDice : {dice:.4f}", fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
print(best_model_path)


In [ ]:
model = AttentionUNetHRNetW32(num_classes=2, pretrained=False, in_channels=IN_CHANNELS).to(DEVICE)   # CHANGED
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pth"), map_location=DEVICE))
model.eval()


In [ ]:
idx = min(566, len(test_dataset) - 1)   # CHANGED: guard against dataset size shrinking after modality filtering
visualize_prediction(model, test_dataset, sample_idx=idx)
